In [ ]:
from tensorflow.keras.applications import MobileNetV2

base = MobileNetV2(
    weights='imagenet', # start from filters learned on ImageNet
    include_top=False, # drop the original 1000-class head
    input_shape=(96, 96, 3) # image size going in
)
base.trainable = False  # freeze: training can't change these weights

print(base.output_shape)
print(base.count_params())

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
(None, 3, 3, 1280)
2257984


In [3]:
from tensorflow.keras import layers, Sequential

tl_model = Sequential([
    layers.Input(shape=(32, 32, 3)),            # one CIFAR image
    layers.Resizing(96, 96),                    # 32x32 -> 96x96
    layers.Rescaling(scale=1/127.5, offset=-1), # 0..255 -> -1..1
    base,                                       # frozen MobileNetV2 from before
    layers.GlobalAveragePooling2D(),            # 3x3x1280 -> 1280 numbers
    layers.Dense(10, activation='softmax')      # 10 CIFAR classes
])

tl_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resizing (Resizing)             │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_96             │ (None, 3, 3, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)